# VariantScope: Dual-Engine Transformer Training
## Genomic and Proteomic Variant Effect Prediction
### Kaggle T4 GPU Training Pipeline

In [ ]:
!pip install -q torch transformers peft accelerate datasets safetensors
!pip install -q huggingface_hub sentencepiece biopython scikit-learn matplotlib seaborn

In [ ]:
import os
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'  # Set via Kaggle Secrets or env
os.environ['HF_MODEL_REPO'] = 'Arko007/variantscope-models'
os.environ['KAGGLE_USERNAME'] = 'anamitrasarkar007'
os.environ['KAGGLE_KEY'] = '3f52d6e5ef8f6cc9c61d697b4941f972'

In [ ]:
!git clone https://github.com/Anamitra-Sarkar/variant-scope.git
%cd variant-scope/training

## Phase 1: ESM-2 + LoRA Fine-Tuning

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from sklearn.metrics import roc_auc_score, accuracy_score
from huggingface_hub import HfApi
import os

HF_TOKEN = os.environ['HF_TOKEN']
HF_REPO = os.environ['HF_MODEL_REPO']

MODEL_NAME = 'facebook/esm2_t30_150M_UR50D'

print('Loading ESM-2 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print('Loading ESM-2 model...')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, trust_remote_code=True)

peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1, target_modules=['query', 'value', 'key', 'dense'])
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Create synthetic dataset (replace with real ClinVar data in production)
def create_dataset(n=500, seq_len=256):
    np.random.seed(42)
    aa = list('ACDEFGHIKLMNPQRSTVWY')
    seqs = [''.join(np.random.choice(aa, size=seq_len)) for _ in range(n)]
    labels = [1 if np.random.random() > 0.7 else 0 for _ in range(n)]
    return seqs, labels

seqs, labels = create_dataset(500, 128)

enc = tokenizer(seqs, truncation=True, padding=True, max_length=512)
enc['label'] = labels
dataset = Dataset.from_dict(enc)

args = TrainingArguments(
    output_dir='./esm2_finetuned',
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
    save_total_limit=2,
    logging_steps=5
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    eval_dataset=dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print('Starting ESM-2 training...')
trainer.train()

trainer.save_model('./esm2_finetuned')
tokenizer.save_pretrained('./esm2_finetuned')

api = HfApi()
api.create_repo(f'{HF_REPO}/esm2-lora', exist_ok=True, token=HF_TOKEN)
api.upload_folder(folder_path='./esm2_finetuned', repo_id=f'{HF_REPO}/esm2-lora', repo_type='model', token=HF_TOKEN)
print(f'Uploaded ESM-2 model to {HF_REPO}/esm2-lora')

## Phase 2: DNABERT-2 + LoRA Fine-Tuning

In [ ]:
MODEL_NAME = 'zhihan1996/DNABERT-2-117M'

print('Loading DNABERT-2...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, trust_remote_code=True)

peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1, target_modules=['query', 'value'])
model = get_peft_model(model, peft_config)

# Create genomic dataset
import random
bases = ['A', 'C', 'G', 'T']
dna_seqs = [''.join(random.choices(bases, k=256)) for _ in range(500)]
dna_labels = [1 if random.random() > 0.7 else 0 for _ in range(500)]

enc = tokenizer(dna_seqs, truncation=True, padding=True, max_length=512)
enc['label'] = dna_labels
dna_dataset = Dataset.from_dict(enc)

args = TrainingArguments(
    output_dir='./dnabert2_finetuned',
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    save_total_limit=2,
    logging_steps=5
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dna_dataset,
    eval_dataset=dna_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print('Starting DNABERT-2 training...')
trainer.train()

trainer.save_model('./dnabert2_finetuned')
tokenizer.save_pretrained('./dnabert2_finetuned')

api = HfApi()
api.create_repo(f'{HF_REPO}/dnabert2-lora', exist_ok=True, token=HF_TOKEN)
api.upload_folder(folder_path='./dnabert2_finetuned', repo_id=f'{HF_REPO}/dnabert2-lora', repo_type='model', token=HF_TOKEN)
print(f'Uploaded DNABERT-2 model to {HF_REPO}/dnabert2-lora')

## Phase 3: Sparse Autoencoder Training

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, sparsity_lambda=1e-3):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)
        self.sparsity_lambda = sparsity_lambda

    def forward(self, x):
        encoded = self.encoder(x)
        sparse = F.relu(encoded)
        reconstructed = self.decoder(sparse)
        recon_loss = F.mse_loss(reconstructed, x)
        sparsity_loss = self.sparsity_lambda * sparse.mean(dim=0).abs().sum()
        return recon_loss + sparsity_loss, sparse

# Extract embeddings from frozen ESM-2
from transformers import AutoModel
esm = AutoModel.from_pretrained('facebook/esm2_t30_150M_UR50D', trust_remote_code=True).cuda().eval()

seqs, _ = create_dataset(200, 128)
embeddings = []
with torch.no_grad():
    for i in range(0, len(seqs), 16):
        inputs = tokenizer(seqs[i:i+16], return_tensors='pt', padding=True, truncation=True, max_length=512)
        inputs = {k: v.cuda() for k, v in inputs.items()}
        outputs = esm(**inputs)
        emb = outputs.last_hidden_state[:, 0, :].cpu()
        embeddings.append(emb)

embeddings = torch.cat(embeddings, dim=0)
print(f'Embedding shape: {embeddings.shape}')

sae = SparseAutoencoder(input_dim=embeddings.shape[1], hidden_dim=256).cuda()
optimizer = torch.optim.AdamW(sae.parameters(), lr=1e-3)

best_loss = float('inf')
patience_counter = 0

for epoch in range(50):
    total_loss = 0
    for i in range(0, len(embeddings), 32):
        batch = embeddings[i:i+32].cuda()
        loss, _ = sae(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / (len(embeddings) // 32 + 1)
    print(f'Epoch {epoch+1}/50 - Loss: {avg_loss:.6f}')

    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(sae.state_dict(), './sae.pt')
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f'Early stopping at epoch {epoch+1}')
            break

api = HfApi()
api.create_repo(f'{HF_REPO}/sae', exist_ok=True, token=HF_TOKEN)
api.upload_file(path_or_fileobj='./sae.pt', path_in_repo='sae.pt', repo_id=f'{HF_REPO}/sae', repo_type='model', token=HF_TOKEN)
print(f'Uploaded SAE to {HF_REPO}/sae')

## Training Complete
All models uploaded to HuggingFace: https://huggingface.co/Arko007/variantscope-models